# 5.1 Code Brief: Course Bottleneck Detection with Unsupervised Learning

Condensed reference for the unsupervised learning pipeline, applied to course-section data.

## Load the Data

```python
from google.colab import drive
drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
df = pd.read_csv(f'{project_path}/data/bottleneck.csv')   # 1,000 course sections x 12 vars
```

## The Universal Pipeline

```python
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 1. Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# 2. Find optimal k (Silhouette)
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    print(f"k={k}: silhouette={silhouette_score(X_scaled, labels):.3f}")

# 3. Fit K-Means
km = KMeans(n_clusters=4, n_init=10, random_state=42)
df['Cluster'] = km.fit_predict(X_scaled)

# 4. PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'], cmap='Set2')

# 5. Profile clusters
profile = df.groupby('Cluster').mean()
```

## Choosing k: silhouette narrows, interpretability decides

On this data the silhouette peaks at **k = 3** (≈ 0.38) but we use **k = 4** (≈ 0.28). Why:

- The **bottleneck cluster is identical at k=3 and k=4** — the same ~200 high-DFW/low-pass
  sections. The answer to "which courses are bottlenecks?" doesn't depend on k.
- k=4 splits the healthy bucket into two useful types (upper-division vs. core); k=3 merges them.
- Splitting a blob into two adjacent real sub-groups always dips silhouette a little — expected, not a red flag.

**Rule:** use the silhouette score to narrow the field, then let cluster *meaning* pick the final k.

```python
# Sanity-check the choice: compare profiles at k=3 vs k=4
for k in (3, 4):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    print(f"k={k}", df.groupby(lab)[['DFW_RATE', 'PASS_RATE', 'PREREQUISITE_COUNT']].mean().round(2), sep="\n")
```

## Risk Prioritization Index

```python
df['RISK_INDEX'] = (
    0.4 * (df['DFW_RATE'] / df['DFW_RATE'].max()) +
    0.3 * (df['REPEAT_RATE'] / df['REPEAT_RATE'].max()) +
    0.3 * ((1 - df['PASS_RATE']) / (1 - df['PASS_RATE']).max())
)
top_risk = df.nlargest(20, 'RISK_INDEX')
```